# 02b — MAUDE Safety Reference & Mapping (Exploratory)

This notebook is an *optional*, exploratory companion to the core Phase 2 pipeline.  
Its purpose is **not** to change the main feature-engineering flow, but to sit beside it as a safety-signal reference and design space.

**Goals of this notebook:**

- Load the Phase 2 scenario table  
  `data/interim/trials_scenarios.parquet` (one row per trial).
- Load the brand-level MAUDE safety summary  
  `data/external/maude_safety_summary.csv` (one row per device brand).
- Inspect both tables side by side:
  - What do trial sponsors and regions look like?
  - What do MAUDE brands and their `total_events` / `safety_score` look like?
- Begin exploring *qualitative* relationships between:
  - clinical trial sponsors / conditions, and  
  - device brands with high MAUDE event volumes.

At this stage, we deliberately **do not** force a hard join between MAUDE and the trial scenarios. Instead, this notebook acts as a sandbox to understand the shapes of both datasets and to inform any future, carefully designed safety-aware mappings or scenarios.

In [1]:
# ============================================================
# Cell 1 — Load trials_scenarios and MAUDE brand safety summary
# ============================================================

from pathlib import Path
import pandas as pd

def log(msg: str) -> None:
    print(msg)

# Paths to existing artifacts
SCENARIOS_PARQUET = Path("data/interim/trials_scenarios.parquet")
SAFETY_CSV = Path("data/external/maude_safety_summary.csv")

# --- Load trials_scenarios ----------------------------------
if SCENARIOS_PARQUET.exists():
    trials_scenarios = pd.read_parquet(SCENARIOS_PARQUET)
    log(f"[Cell 1] Loaded trials_scenarios with shape {trials_scenarios.shape}")
else:
    raise FileNotFoundError(
        f"[Cell 1] Missing {SCENARIOS_PARQUET}. "
        "Run 02_scenario_prep_and_risk_features.ipynb first."
    )

# --- Load MAUDE brand-level safety (optional) ---------------
if SAFETY_CSV.exists():
    maude_safety = pd.read_csv(SAFETY_CSV)
    log(
        f"[Cell 1] Loaded MAUDE safety summary from {SAFETY_CSV} "
        f"with shape {maude_safety.shape}"
    )
else:
    log(
        f"[Cell 1] WARNING: {SAFETY_CSV} not found. "
        "Safety analyses will be skipped."
    )
    maude_safety = None

trials_scenarios.head()


[Cell 1] Loaded trials_scenarios with shape (557292, 12)
[Cell 1] Loaded MAUDE safety summary from data/external/maude_safety_summary.csv with shape (155489, 3)


,nct_id,brief_title,overall_status,phase,conditions,interventions,location_countries,lead_sponsor,lead_sponsor_norm,region_label,estimated_trial_cost,enrollment_feasibility_score
0,NCT00000102,Congenital Adrenal Hyperplasia: Calcium Channe...,Completed,Phase 1/Phase 2,[Congenital Adrenal Hyperplasia],[Nifedipine],[United States],National Center for Research Resources (NCRR),National Center for Research Resources (NCRR),Global / Multi-Region,1.5,1.0
1,NCT00000104,Does Lead Burden Alter Neuropsychological Deve...,Completed,None,[Lead Poisoning],[ERP measures of attention and memory],[United States],National Center for Research Resources (NCRR),National Center for Research Resources (NCRR),Global / Multi-Region,1.0,1.0
2,NCT00000105,Vaccination With Tetanus and KLH to Assess Imm...,Terminated,None,[Cancer],"[Intracel KLH Vaccine, Biosyn KLH, Montanide I...",[United States],"Masonic Cancer Center, University of Minnesota","Masonic Cancer Center, University of Minnesota",Global / Multi-Region,1.0,1.0
3,NCT00000106,41.8 Degree Centigrade Whole Body Hyperthermia...,Unknown status,N/A,[Rheumatic Diseases],[Whole body hyperthermia unit],[United States],National Center for Research Resources (NCRR),National Center for Research Resources (NCRR),Global / Multi-Region,1.0,1.0
4,NCT00000107,Body Water Content in Cyanotic Congenital Hear...,Completed,None,"[Heart Defects, Congenital]",[],[United States],National Center for Research Resources (NCRR),National Center for Research Resources (NCRR),Global / Multi-Region,1.0,1.0


### What Cell 1 Just Did

In this step, we loaded the two core Phase 2 artifacts into memory:

1. **`trials_scenarios` (trial-level scenario table)**  
   - Read from `data/interim/trials_scenarios.parquet`.  
   - One row per `nct_id`, already de-duplicated in the main Phase 2 notebook.  
   - Brought in the key fields we use to reason about trials, including:
     - Identification and design: `nct_id`, `brief_title`, `overall_status`, `phase`
     - Clinical framing: `conditions`, `interventions`, `location_countries`
     - Sponsor and geography: `lead_sponsor`, `lead_sponsor_norm`, `region_label`
     - Operational features: `estimated_trial_cost`, `enrollment_feasibility_score`

2. **`maude_safety` (brand-level MAUDE safety reference)**  
   - Read from `data/external/maude_safety_summary.csv` when available.  
   - One row per normalized device brand, with:
     - `brand`
     - `total_events` (unique MAUDE events per brand)
     - `safety_score` (relative burden of events scaled to \[0, 1\])

If the scenarios parquet was missing, the cell failed fast with a clear error so we don’t analyze without a valid scenario table. If the MAUDE safety file was missing, the cell logged a warning and continued with `maude_safety = None`, allowing the rest of the notebook to run while simply skipping safety-focused exploration.

In [2]:
# ============================================================
# Cell 2 — Inspect MAUDE safety reference table
# ============================================================

if maude_safety is None:
    log("[Cell 2] No MAUDE safety table loaded; skipping inspection.")
else:
    log("[Cell 2] Basic info for maude_safety:")
    log(f"Shape: {maude_safety.shape}")
    log(f"Columns: {list(maude_safety.columns)}")
    
    display(maude_safety.head(10))

    # Dtypes + simple stats on total_events and safety_score
    dtypes_summary = maude_safety.dtypes.to_frame(name="dtype")
    display(dtypes_summary)

    if "total_events" in maude_safety.columns:
        log("[Cell 2] total_events summary:")
        display(maude_safety["total_events"].describe())

    if "safety_score" in maude_safety.columns:
        log("[Cell 2] safety_score summary:")
        display(maude_safety["safety_score"].describe())



[Cell 2] Basic info for maude_safety:
Shape: (155489, 3)
Columns: ['brand', 'total_events', 'safety_score']


,brand,total_events,safety_score
0,!!! POWERFLEXX,1,0.000001
1,!!93-P PROFLEXX,1,0.000001
2,!M1,1,0.000001
3,"""1.0MM"" SYSTEM TWIST DRILL TW 7X50MM 3MMSTOP W...",1,0.000001
4,"""1.5MM"" SYSTEM 1.5X4MM HT SD X-DR SCR 5-PK",1,0.000001
5,"""1.5MM"" SYSTEM 1.5X6MM HT SD X-DR SCR",7,0.000008
6,"""1.5MM"" SYSTEM PLATE 1.5 2 HOLE LONG STRAIGHT",1,0.000001
7,"""1.5MM"" SYSTEM TW DRILL 1.1X50MM 15MMSTP W/NT",1,0.000001
8,"""1.5MM"" SYSTEM TW DRILL 1.1X50MM 7MMSTOP W/NT",2,0.000002
9,"""1.5MM"" SYSTEM TWIST DRILL",2,0.000002


,dtype
brand,object
total_events,int64
safety_score,float64


[Cell 2] total_events summary:


count    155489.000000
mean         63.825750
std        3336.255216
min           1.000000
25%           1.000000
50%           1.000000
75%           4.000000
max      894586.000000
Name: total_events, dtype: float64

[Cell 2] safety_score summary:


count    155489.000000
mean          0.000071
std           0.003729
min           0.000001
25%           0.000001
50%           0.000001
75%           0.000004
max           1.000000
Name: safety_score, dtype: float64

### What Cell 2 Just Did

This step sanity-checked the MAUDE safety reference table so we understand its structure before trying to use it.

If `maude_safety` was available, the cell:

- Reported the **overall shape** of the table and its **column names**, confirming that we have the expected fields (`brand`, `total_events`, `safety_score`).
- Displayed the **first few rows**, giving a concrete feel for how brands and event counts are represented (including some long or noisy brand strings).
- Printed a small **dtypes summary**, so we can see how pandas interpreted each column (e.g., numeric vs. object).
- Summarized the distributions of:
  - `total_events` — how many MAUDE events are associated with each brand.
  - `safety_score` — the relative scaling of those event counts into the \[0, 1\] range.

If `maude_safety` was not loaded (file missing), the cell simply logged that situation and skipped inspection, keeping the notebook runnable even when safety data is absent.


In [3]:
# ============================================================
# Cell 3 — Explore sponsors and MAUDE brands (no joins yet)
# ============================================================

log("[Cell 3] Top lead sponsors in trials_scenarios:")
display(
    trials_scenarios["lead_sponsor"]
    .value_counts()
    .head(20)
    .to_frame(name="trial_count")
)

if maude_safety is not None and "brand" in maude_safety.columns:
    log("[Cell 3] Sample of MAUDE brands:")
    display(
        maude_safety["brand"]
        .head(20)
        .to_frame(name="brand")
    )
else:
    log("[Cell 3] MAUDE brands not available for exploration.")


[Cell 3] Top lead sponsors in trials_scenarios:


,trial_count
lead_sponsor,
Assiut University,4348
Cairo University,4125
GlaxoSmithKline,3569
National Cancer Institute (NCI),3513
AstraZeneca,3347
Assistance Publique - Hôpitaux de Paris,3329
Pfizer,3211
Mayo Clinic,3110
M.D. Anderson Cancer Center,2925


[Cell 3] Sample of MAUDE brands:


,brand
0,!!! POWERFLEXX
1,!!93-P PROFLEXX
2,!M1
3,"""1.0MM"" SYSTEM TWIST DRILL TW 7X50MM 3MMSTOP W..."
4,"""1.5MM"" SYSTEM 1.5X4MM HT SD X-DR SCR 5-PK"
5,"""1.5MM"" SYSTEM 1.5X6MM HT SD X-DR SCR"
6,"""1.5MM"" SYSTEM PLATE 1.5 2 HOLE LONG STRAIGHT"
7,"""1.5MM"" SYSTEM TW DRILL 1.1X50MM 15MMSTP W/NT"
8,"""1.5MM"" SYSTEM TW DRILL 1.1X50MM 7MMSTOP W/NT"
9,"""1.5MM"" SYSTEM TWIST DRILL"


### What Cell 3 Just Did

This step took a first, qualitative look at how the clinical trial sponsors and MAUDE brands compare, without forcing any joins.

From the **trial side**, it:

- Listed the top `lead_sponsor` values in `trials_scenarios`, along with how many trials each sponsor appears in.
- Provides a quick sense of which organizations dominate the current scenario table (major pharma, academic centers, large health systems, etc.).

From the **MAUDE side**, it:

- Displayed a small sample of `brand` values from `maude_safety` (when available).
- Gave a feel for how noisy or granular the device brand strings are (e.g., long descriptions, model variants, or marketing names).

Taken together, this cell is purely exploratory: it helps calibrate expectations about the gap between **sponsor-level trial metadata** and **brand-level device safety signals**, and sets the stage for any future, carefully designed mapping strategies.

## Notebook Summary — MAUDE Safety as a Reference Layer

This exploratory notebook does not alter the core Phase 2 pipeline. Instead, it treats MAUDE device safety as a separate reference layer that can inform future safety-aware scenarios.

In this notebook, we:

1. Loaded the Phase 2 trial scenario table  
   `data/interim/trials_scenarios.parquet`  
   bringing in one row per `nct_id` with sponsor, phase, condition, region, cost, and enrollment feasibility features.

2. Loaded the brand-level MAUDE safety summary  
   `data/external/maude_safety_summary.csv`  
   containing one row per normalized device brand with `total_events` and a relative `safety_score` on \[0, 1\].

3. Inspected both datasets side by side:  
   - Top sponsors and their trial counts from `trials_scenarios`.  
   - Example MAUDE brands and the distribution of safety metrics from `maude_safety`.

This notebook’s role is to clarify the shapes and semantics of the trial and safety tables, and to highlight the conceptual gap between sponsor-level metadata and brand-level device signals. It provides a foundation for designing future, explicit mapping strategies without coupling MAUDE logic directly into the main Phase 2 feature-engineering notebook.
